In [19]:
from unittest import result

from debugpy.launcher import output
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-haiku-4-5"

In [20]:
# Helper functions
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)


def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message.content[0].text

In [21]:
import json


def generate_dataset():
    prompt = """
Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
    {
        "task": "Description of task",
    },
    ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""

    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")
    text = chat(messages, stop_sequences=["```"])
    return json.loads(text)



In [22]:
# Call the function to execute it
dataset = generate_dataset()
dataset

[{'task': "Write a Python function that extracts the AWS account ID from an ARN string. For example, 'arn:aws:s3:::my-bucket' should return None, but 'arn:aws:iam::123456789012:user/john' should return '123456789012'."},
 {'task': "Create a JSON object that represents an AWS IAM policy allowing read-only access to all objects in an S3 bucket named 'my-data-bucket'."},
 {'task': "Write a regular expression that matches valid AWS EC2 instance IDs. Valid instance IDs start with 'i-' followed by exactly 17 hexadecimal characters (e.g., 'i-0a1b2c3d4e5f6g7h8')."}]

In [23]:
with open("dataset.json", "w") as f:
    json.dump(dataset, f, indent=4)

In [24]:
def run_prompt(test_case):
    """Merges the prompt and test case input, then return the rsult"""
    prompt=f"""
    Please solve the following task:

    {test_case['task']}
    """
    messages = []
    add_user_message(messages, prompt)
    output = chat(messages)
    return output

In [25]:
def run_test_case(test_case):
    """Calls run_prompt and then grades the result"""
    output = run_prompt(test_case)


    score=10

    return{
        "output": output,
        "test_case": test_case,
        "score": score
    }

In [26]:
def run_eval(dataset):
    """Loads the dataset and calls run_test_case with each case"""
    results = []
    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)

    return results

In [27]:
with open("dataset.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)

In [28]:
print(json.dumps(results, indent=2))

[
  {
    "output": "# AWS Account ID Extractor from ARN\n\nHere's a solution with multiple approaches:\n\n## Simple Solution\n\n```python\ndef extract_account_id(arn: str) -> str | None:\n    \"\"\"\n    Extracts the AWS account ID from an ARN string.\n    \n    Args:\n        arn: AWS ARN string\n        \n    Returns:\n        Account ID as string, or None if not present\n        \n    Examples:\n        >>> extract_account_id('arn:aws:s3:::my-bucket')\n        None\n        >>> extract_account_id('arn:aws:iam::123456789012:user/john')\n        '123456789012'\n        >>> extract_account_id('arn:aws:ec2:us-east-1:987654321098:instance/i-1234567890abcdef0')\n        '987654321098'\n    \"\"\"\n    try:\n        parts = arn.split(':')\n        # ARN format: arn:partition:service:region:account-id:resource\n        # Index 4 should contain the account ID\n        if len(parts) >= 5:\n            account_id = parts[4]\n            # Return None if account_id is empty string (like S3 buc